# Other embedders

In [2]:
import lancedb
import pandas as pd

db = lancedb.connect("knowledge_base")

df = pd.read_json("data/jokes.json")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


## Sentence transformer

In [5]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)

class JokeModel(LanceModel):
    jokes: str = model.SourceField() 
    vector: Vector(model.ndims()) = model.VectorField()  


print(f"{model.ndims() = }")

table = db.create_table("jokes_sentence_transformers", schema=JokeModel)
table

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2377.25it/s]


model.ndims() = 384


LanceTable(name='jokes_sentence_transformers', _conn=LanceDBConnection(uri='c:\\Users\\nasrw\\Documents\\GitHub\\ai_ops_code_along_de25\\09_lancedb\\knowledge_base'))

In [6]:
table.add(df)

AddResult(version=2)

In [7]:
table.search().limit(5).to_pandas()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.050497, -0.006426, -0.071011, -0.027726, 0...."
2,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.045366, -0.05297, -0.066901, -0.02893, 0.11..."
4,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0..."


In [8]:
table.search("math jokes").to_pandas().head()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.290241
1,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.300917
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.312815
3,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.339917
4,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.347602


In [9]:
table.search("matematikskämt").to_pandas().head()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.370423
1,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.400166
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.427736
3,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0....",0.438430
4,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0...",0.442392


In [10]:
table.search("kemiskämt").to_pandas().head()

,jokes,vector,_distance
0,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.388733
1,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.391546
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.395784
3,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.417908
4,Never trust an atom—they make up everything.,"[0.059281, -0.023624, -0.089997, -0.071139, 0....",0.443533


# Custom embedder

In [16]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from lancedb.embeddings import TextEmbeddingFunction, get_registry
import numpy as np

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API"),
    max_retries=3,
    timeout=60.0,
)

emb = client.embeddings.create(
    model="nvidia/nemotron-3.5-embed-8b",
    input=["hej"],
    extra_body={"input_type": "passage"},
)
len(emb.data[0].embedding)

4096

In [12]:
from custom_embedders import nvidia_nemotron

model = get_registry().get("nvidia-nemotron").create()

class JokeModel(LanceModel):
    jokes: str = model.SourceField() 
    vector: Vector(model.ndims()) = model.VectorField()  


model.ndims()

4096

In [13]:
table = db.create_table("jokes_nvidia", schema=JokeModel)

table.add(df)

AddResult(version=2)

In [14]:
table.search("programmerare tycker inte om ljus").limit(5).to_pandas()

,jokes,vector,_distance
0,C# programmers love coffee—they're always work...,"[-0.004074, -0.055206, 0.00975, 0.003778, 0.00...",1.439223
1,Why did the Python programmer get bitten? Beca...,"[-0.013878, -0.042084, 0.011299, -0.019653, 0....",1.522183
2,Why do data engineers hate nature? Too many un...,"[-0.006134, -0.076904, 0.011269, -0.00084, 0.0...",1.526139
3,C# devs don’t argue; they just throw exceptions.,"[0.009048, -0.062347, 0.02359, 0.017288, 0.013...",1.539856
4,Why is C# like a musical? It has so many classes.,"[-0.00077, -0.090088, 0.015549, 0.007221, 0.01...",1.578951


In [15]:
table.search("kemi skämt").limit(5).to_pandas()

,jokes,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.005886, -0.082947, 0.019272, 0.000912, 0.02...",1.182106
1,Why did the chemist ground his kids? Because t...,"[0.001008, -0.075012, 0.017822, 0.006462, 0.01...",1.225226
2,Chemists have all the solutions… mostly in bea...,"[0.002228, -0.099121, 0.020493, 0.021805, 0.01...",1.361885
3,"Gold walks into a bar. The bartender says, “Au...","[-0.004944, -0.056854, 0.010658, -0.011421, 0....",1.398793
4,Oxygen and Magnesium got together—OMg!,"[-0.011795, -0.058441, 0.016251, -0.004665, 0....",1.439115
